# Model 2 — 1D Convolutional Neural Network

**Efficiency strategy:** Preload the full normalized (N, 301, 3) dataset into GPU VRAM once. DataLoader then serves raw sequence tensors with zero HDF5 reads per epoch. Batch size raised to 2048 (was 256) — reduces train batches from ~1,372 to ~172 per epoch.

**Architecture:**
```
Input (batch, 301, 3) → permute → (batch, 3, 301)
Conv1d(3→32, k=7) → BN → ReLU → MaxPool(2)    [301→150]
Conv1d(32→64, k=5) → BN → ReLU → MaxPool(2)   [150→75]
Conv1d(64→128, k=3) → BN → ReLU → MaxPool(2)  [75→37]
Conv1d(128→128, k=3) → BN → ReLU → AvgPool    [37→1]
Linear(128→64) → ReLU → Dropout(0.3) → Linear(64→1)
```

**Data:** `data/processed/sequences.h5` — battery-level split: 22 train / 6 test batteries.

In [ ]:
import h5py
import json
import math
import os
import sys
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm, trange

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.dataset import BatterySOHDataset, DEFAULT_TEST_BATTERIES
from src.models.cnn1d import SOHCNN1D

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
H5_PATH      = "../data/processed/sequences.h5"
STATS_PATH   = "../data/processed/norm_stats.json"
CKPT_PATH    = "../results/checkpoints/cnn_best.pt"
METRICS_PATH = "../results/metrics/cnn_history.json"
FIG_DIR      = "../results/figures"

EPOCHS       = 100
BATCH_SIZE   = 2048   # large batch — full dataset in VRAM, zero HDF5 reads
LR           = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE     = 10
RANDOM_SEED  = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)
os.makedirs(os.path.dirname(METRICS_PATH), exist_ok=True)

def set_seeds(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seeds()

## Section 1 — Data Loading + GPU Preload

The full normalized dataset (1.52 GB) fits in the 7.7 GB GPU VRAM. Loading once eliminates all HDF5 reads and gzip decompression from the training loop.

In [ ]:
# Step 1: battery-level split
train_idx, test_idx = BatterySOHDataset.create_splits(H5_PATH)

if not os.path.exists(STATS_PATH):
    tmp = BatterySOHDataset(H5_PATH, indices=train_idx, normalize=False)
    tmp.compute_normalization_stats(STATS_PATH)

with open(STATS_PATH) as f:
    stats = json.load(f)
mean = np.array(stats['mean'], dtype=np.float32)   # (3,)
std  = np.array(stats['std'],  dtype=np.float32)    # (3,)

# Step 2: load full dataset into RAM then push to GPU
print('Loading dataset into RAM...', flush=True)
with h5py.File(H5_PATH, 'r') as f:
    X_all = f['X'][:]           # (N, 301, 3) float32
    y_all = f['y'][:] / 100.0   # scale to [0, 1]
print(f'  X: {X_all.shape}  ({X_all.nbytes / 1024**3:.2f} GB)')

X_all = (X_all - mean) / (std + 1e-8)   # normalize in-place (CPU)

print('Pushing to GPU...', end=' ', flush=True)
X_tensor = torch.from_numpy(X_all).to(DEVICE)      # (N, 301, 3)
y_tensor = torch.from_numpy(y_all).to(DEVICE)
del X_all, y_all
torch.cuda.empty_cache()
print(f'done. VRAM: {torch.cuda.memory_allocated() / 1024**2:.1f} MiB')

# Step 3: split
train_idx_t = torch.from_numpy(train_idx)
test_idx_t  = torch.from_numpy(test_idx)

X_train = X_tensor[train_idx_t]   # stays on GPU
y_train = y_tensor[train_idx_t]
X_test  = X_tensor[test_idx_t]
y_test  = y_tensor[test_idx_t]
del X_tensor, y_tensor

print(f'Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}')
print(f'Test batteries: {DEFAULT_TEST_BATTERIES}')
print(f'VRAM after split: {torch.cuda.memory_allocated() / 1024**2:.1f} MiB')

# Step 4: TensorDataset loaders — zero HDF5 reads during training
# Data already on GPU so pin_memory=False, non_blocking=True in loop
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=0)
test_loader  = DataLoader(TensorDataset(X_test,  y_test),  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0)

print(f'Train batches/epoch: {len(train_loader)}  (was 1372 with batch=256 + HDF5)')
xb, yb = next(iter(train_loader))
print(f'Batch x: {xb.shape}  y range: [{yb.min():.3f}, {yb.max():.3f}]')

## Section 2 — Raw Signal Inspection

Visualise a few steps from the test set to understand what the CNN is learning from.

In [ ]:
xb_test, yb_test = X_test[:BATCH_SIZE].cpu(), y_test[:BATCH_SIZE].cpu()
CHANNEL_NAMES = ["Voltage (V)", "Current (A)", "Temperature (°C)"]
COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c"]

# Pick 4 samples spanning the SOH range
sort_idx = yb_test.argsort()
sample_indices = [sort_idx[0].item(), sort_idx[len(sort_idx)//3].item(),
                  sort_idx[2*len(sort_idx)//3].item(), sort_idx[-1].item()]

fig, axes = plt.subplots(3, 4, figsize=(16, 8), sharex=True)
for col, idx in enumerate(sample_indices):
    soh = yb_test[idx].item() * 100
    for row, (cname, color) in enumerate(zip(CHANNEL_NAMES, COLORS)):
        axes[row, col].plot(xb_test[idx, :, row].numpy(), color=color, lw=1)
        if row == 0:
            axes[row, col].set_title(f"SOH = {soh:.1f}%", fontsize=10)
        if col == 0:
            axes[row, col].set_ylabel(cname, fontsize=9)
        axes[row, col].grid(alpha=0.3)

fig.suptitle("Raw Input Sequences (normalized) — 4 samples at different SOH levels",
             fontsize=12, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "cnn_raw_signals.png"), dpi=150)
plt.show()

## Section 3 — Model Architecture

In [ ]:
set_seeds()
model = SOHCNN1D().to(DEVICE)
print(model)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {n_params:,}")

# Trace intermediate shapes
dummy = torch.randn(4, 301, 3).to(DEVICE)
with torch.no_grad():
    x = dummy.permute(0, 2, 1)
    print(f"\nInput:                      {tuple(dummy.shape)}")
    print(f"After permute:              {tuple(x.shape)}")
    x = model.pool1(model.conv1(x)); print(f"After conv1 + pool1:        {tuple(x.shape)}")
    x = model.pool2(model.conv2(x)); print(f"After conv2 + pool2:        {tuple(x.shape)}")
    x = model.pool3(model.conv3(x)); print(f"After conv3 + pool3:        {tuple(x.shape)}")
    x = model.global_pool(model.conv4(x)); print(f"After conv4 + global_pool:  {tuple(x.shape)}")
    out = model(dummy); print(f"Output:                     {tuple(out.shape)}")

## Section 4 — Training

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total = 0.0
    bar = tqdm(loader, desc='Training', leave=True, unit='batch')
    for x, y in bar:
        # Data already on GPU
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(x).squeeze(1), y)
        loss.backward()
        optimizer.step()
        total += loss.item()
        bar.set_postfix(loss=f'{loss.item():.4f}')
    return total / len(loader)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, targets = [], []
    bar = tqdm(loader, desc='Validating', leave=True, unit='batch')
    for x, y in bar:
        preds.append(model(x).squeeze(1).cpu())
        targets.append(y.cpu())
    preds   = torch.cat(preds).numpy() * 100.0
    targets = torch.cat(targets).numpy() * 100.0
    mae  = float(np.abs(preds - targets).mean())
    rmse = float(math.sqrt(((preds - targets)**2).mean()))
    return mae, rmse, preds, targets

In [ ]:
set_seeds()
model = SOHCNN1D().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=5, factor=0.5)
criterion = nn.MSELoss()

history    = {"train_loss": [], "val_mae": [], "val_rmse": []}
best_val_mae = float("inf")
no_improve   = 0

print(f'Batch size: {BATCH_SIZE}  |  Train batches/epoch: {len(train_loader)}')
print(f'VRAM before training: {torch.cuda.memory_allocated() / 1024**2:.1f} MiB')

for epoch in range(1, EPOCHS + 1):
    print(f'\n{"="*60}')
    print(f'Epoch {epoch}/{EPOCHS}')
    print(f'{"="*60}')

    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_mae, val_rmse, _, _ = evaluate(model, test_loader)
    scheduler.step(val_mae)

    history["train_loss"].append(train_loss)
    history["val_mae"].append(val_mae)
    history["val_rmse"].append(val_rmse)

    lr_now = optimizer.param_groups[0]['lr']
    vram   = torch.cuda.memory_allocated() / 1024**2

    print(f'\nTrain Loss: {train_loss:.4f}')
    print(f'Val MAE:    {val_mae:.4f}%')
    print(f'Val RMSE:   {val_rmse:.4f}%')
    print(f'LR:         {lr_now:.6f}')
    print(f'VRAM:       {vram:.0f} MiB')

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        no_improve   = 0
        torch.save({"epoch": epoch, "val_mae": val_mae,
                    "model_state_dict": model.state_dict()}, CKPT_PATH)
        print(f'Model saved to {CKPT_PATH}')
        print('✓ New best model saved!')
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'Early stopping at epoch {epoch}.')
            break

with open(METRICS_PATH, "w") as f:
    json.dump(history, f, indent=2)
print(f'\nBest val MAE: {best_val_mae:.4f}%  |  Checkpoint: {CKPT_PATH}')

## Section 5 — Results

In [ ]:
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Loaded best checkpoint (epoch {ckpt['epoch']}, val MAE {ckpt['val_mae']:.4f}%)")

test_mae, test_rmse, preds, targets = evaluate(model, test_loader)
print(f"\nTest MAE:  {test_mae:.4f} SOH%")
print(f"Test RMSE: {test_rmse:.4f} SOH%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history["train_loss"], color="#DD8452", lw=1.5)
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE Loss")
axes[0].set_title("Training Loss"); axes[0].grid(alpha=0.3)

axes[1].plot(history["val_mae"],  label="Val MAE",  color="darkorange", lw=1.5)
axes[1].plot(history["val_rmse"], label="Val RMSE", color="crimson",    lw=1.5, ls="--")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("SOH%")
axes[1].set_title("Validation Metrics"); axes[1].legend(); axes[1].grid(alpha=0.3)

fig.suptitle("1D-CNN — Training Curves", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "cnn_training_curves.png"), dpi=150)
plt.show()

In [ ]:
errors = preds - targets
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

lims = [min(targets.min(), preds.min()) - 2, max(targets.max(), preds.max()) + 2]
axes[0].scatter(targets, preds, alpha=0.25, s=5, color="#DD8452")
axes[0].plot(lims, lims, "k--", lw=1)
axes[0].set_xlim(lims); axes[0].set_ylim(lims)
axes[0].set_xlabel("Actual SOH (%)"); axes[0].set_ylabel("Predicted SOH (%)")
axes[0].set_title(f"Predicted vs Actual  MAE={test_mae:.2f}%  RMSE={test_rmse:.2f}%")
axes[0].grid(alpha=0.3)

axes[1].hist(errors, bins=60, color="#DD8452", edgecolor="white", alpha=0.85)
axes[1].axvline(0,              color="black", lw=1.2, ls="--", label="Zero error")
axes[1].axvline(errors.mean(),  color="red",   lw=1.5, label=f"Mean={errors.mean():.2f}%")
axes[1].set_xlabel("Prediction Error (SOH%)"); axes[1].set_ylabel("Count")
axes[1].set_title(f"Error Distribution  std={errors.std():.2f}%")
axes[1].legend(); axes[1].grid(axis="y", alpha=0.3)

fig.suptitle("1D-CNN — Test Set Results", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "cnn_results.png"), dpi=150)
plt.show()

In [ ]:
# Visualise conv1 filter activations for one input step
model.eval()
sample = X_test[:1]   # already on GPU

with torch.no_grad():
    act = model.conv1(sample.permute(0, 2, 1))   # (1, 32, 301)

act_np = act[0].cpu().numpy()   # (32, 301)

fig, axes = plt.subplots(4, 8, figsize=(18, 6), sharex=True)
for i, ax in enumerate(axes.flatten()):
    ax.plot(act_np[i], lw=0.8, color="#DD8452")
    ax.set_title(f"F{i}", fontsize=7)
    ax.axis("off")

fig.suptitle("Conv1 Filter Activations (32 filters × 301 timesteps)  SOH={:.1f}%".format(
    y_test[0].item() * 100), fontsize=11, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "cnn_feature_maps.png"), dpi=150)
plt.show()

In [ ]:
print("=" * 40)
print("1D-CNN — FINAL SUMMARY")
print("=" * 40)
print(f"Parameters : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Batch size : {BATCH_SIZE}  (batches/epoch: {len(train_loader)})")
print(f"Test MAE   : {test_mae:.4f} SOH%")
print(f"Test RMSE  : {test_rmse:.4f} SOH%")
print(f"Error mean : {errors.mean():.4f} SOH%")
print(f"Error std  : {errors.std():.4f} SOH%")
print(f"Error p95  : {np.percentile(np.abs(errors), 95):.4f} SOH%")